<a href="https://colab.research.google.com/github/dhaev/Data-projects/blob/main/freight_analysis/freight_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install diskcache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00


In [2]:
import os, glob
import time
import json
import sqlite3
import hashlib
from diskcache import Cache
import requests
import math
import plotly.express as px
import pandas as pd
import numpy as np
from datetime import date, timedelta

In [3]:
pd.set_option('display.max_columns', None)       # Show all columns
pd.set_option('display.width', None)             # Auto-detect terminal width
pd.set_option('display.max_colwidth', None)      # Show full content in each column

In [4]:
OPENROUTESERVICE_API_KEY = "eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjBjZjE2MDQ5YTgzOTRjNDJhYjA3NzI3OWZkODY3MDY1IiwiaCI6Im11cm11cjY0In0=" # Replace with your actual ORS API key
OPENROUTESERVICE_API_URL = "https://api.openrouteservice.org/v2/directions/driving-hgv/geojson"
ORS_ROUTE_CACHE = Cache("/content/drive/MyDrive/freight_analysis/cache/ors_routes_cache")
DRIVING_HOURS = 10
SECONDS_TO_HOURS = 3600

In [5]:
# --- Database Setup ---
# Define the database file name
DB_FILE = '/content/drive/MyDrive/freight_analysis/freight.db'
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

In [6]:
#Get route info
# --- Helper function to get route geometry from OpenRouteService ---
def get_route_geometry(start_lon, start_lat, end_lon, end_lat, api_key=OPENROUTESERVICE_API_KEY):
    """
    Fetches route geometry (list of [lat, lon] points) from OpenRouteService.
    Returns None if the route cannot be found or an error occurs.
    Caches results to reduce API calls using diskcache.
    """
    cache_key = (start_lon, start_lat, end_lon, end_lat)
    cached_result = ORS_ROUTE_CACHE.get(cache_key)
    if cached_result is not None: # Check for None explicitly, as a valid route could be an empty list if ORS returns no geometry
        # print(f"Fetching ORS route from disk cache for {cache_key}")
        return cached_result

    headers = {
        'Accept': 'application/json, application/geo+json, application/gpx+xml, img/png; charset=utf-8',
        'Authorization': api_key,
        'Content-Type': 'application/json; charset=utf-8'
    }
    # Coordinates format for ORS is [longitude, latitude]
    body = {
        "coordinates": [[start_lon, start_lat], [end_lon, end_lat]]
    }

    try:
        time.sleep(7)
        response = requests.post(OPENROUTESERVICE_API_URL, headers=headers, json=body)#, timeout=60)
        # response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)
        data = response.json()
        status = response.status_code
        print(status)

        # Extract coordinates from the GeoJSON response
        if status == 200 and data and 'features' in data and len(data['features']) > 0:
          feature = data['features'][0]
          geometry = feature['geometry']['coordinates']
          summary = feature['properties']['summary']

          distance = float(summary['distance']) * 0.000621371
          duration_in_seconds = float(summary['duration'])
          duration_in_hours = duration_in_seconds / SECONDS_TO_HOURS
          duration_in_driving_days = duration_in_hours / DRIVING_HOURS

          plotly_geometry = [[point[1], point[0]] for point in geometry]

          cache_value = {
              'status': status,
              'geometry': geometry,
              'plotly_geometry': plotly_geometry,
              'distance': distance,
              'duration_in_seconds': duration_in_seconds,
              'duration_in_hours': duration_in_hours,
              'duration_in_driving_days': duration_in_driving_days
          }
          ORS_ROUTE_CACHE.set(cache_key, cache_value)
          return cache_value
        else:
            print(f"No route features found for {start_lat},{start_lon} to {end_lat},{end_lon}  [[{start_lon},{start_lat}],[{end_lon},{end_lat}] ]")
            ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result
            return {'status': status}
    except requests.exceptions.RequestException as e:
        print(f"Error fetching route from OpenRouteService: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response status: {e.response.status_code}")
            print(f"Response body: {e.response.text}")
        ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result on error
        return None
    except json.JSONDecodeError:
        print(f"Error decoding JSON response from OpenRouteService for route {start_lat},{start_lon} to {end_lat},{end_lon} [[{start_lon},{start_lat}],[{end_lon},{end_lat}] ]")
        ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result on error
        return None
    except Exception as e:
        print(f"An unexpected error occurred in get_route_geometry: {e}")
        ORS_ROUTE_CACHE.set(cache_key, None) # Cache None result on error
        return None

Store all equipment(truck types) as unique values in an sqlite database

In [7]:
# --- Helper Function for Equipment ID Management ---
def get_or_create_equipment_id(equipment_name, cursor, conn):
    """
    Checks if an equipment name exists in the 'equipments' table.
    If it exists, returns its ID. If not, inserts it and returns the new ID.
    """
    if equipment_name is None:
        return None

    # Try to find the equipment by name
    cursor.execute("SELECT id FROM equipments WHERE equipment = ?", (equipment_name,))
    result = cursor.fetchone()

    if result:
        # Equipment found, return its ID
        return result[0]
    else:
        # Equipment not found, insert it
        cursor.execute("INSERT INTO equipments (equipment) VALUES (?)", (equipment_name,))
        conn.commit() # Commit the insert operation
        # Get the ID of the last inserted row
        return cursor.lastrowid

## get_nextload_data()

In [8]:
def get_nextload_data():
  extracted_data =  []
  folder_path = '/content/drive/MyDrive/freight_analysis/loadReq'
  if os.path.exists(folder_path):
      for f in glob.glob(folder_path + '/*.json'):
          try:
              # Open and load the JSON file
              with open(str(f), 'r', encoding='utf-8') as file:
                  nextload_data = json.load(file)
          except FileNotFoundError:
              print(f"Error: The JSON file '{json_file_path}' was not found. Please ensure it's in the correct directory.")
          except json.JSONDecodeError as e:
              print(f"Error decoding JSON from '{json_file_path}': {e}")
          except sqlite3.Error as e:
              print(f"SQLite database error: {e}")
          except Exception as e:
              print(f"An unexpected error occurred: {e}")


          loads = nextload_data.get('loads', []) # Safely get the 'loads' list
          # Keep track of all unique equipment display names encountered for testing
          all_processed_equipment_names = set()

          for load in loads:
              # Extract equipment display names
              equipment_display_names = [
                  eq_type.get("displayName")
                  for eq_type in load.get("equipmentTypes", [])
                  if eq_type.get("displayName") is not None # Ensure displayName exists
              ]

              # Convert equipment display names to database IDs
              equipment_type_ids = []
              for eq_name in equipment_display_names:
                  eq_id = get_or_create_equipment_id(eq_name, cursor, conn)
                  if eq_id is not None:
                      equipment_type_ids.append(eq_id)
                      all_processed_equipment_names.add(eq_name) # Add to set for testing

              # Build the dictionary for the current load, using .get() for safe access
              extracted_data.append({
                  "source": "nextload",
                  "source_id": load.get("id"),
                  "load_reference": load.get("referenceNumber"),
                  "post_date": load.get("originalPostingDate"),
                  "last_updated": load.get("postingDate"), # Corrected key to 'postingDate' as per JSON snippet

                  "pickup_id": load.get("pick", {}).get("id", {}),
                  "pickup_city": load.get("pick", {}).get("location", {}).get("city"),
                  "pickup_state": load.get("pick", {}).get("location", {}).get("state"),
                  "pickup_country": load.get("pick", {}).get("location", {}).get("country"),
                  "pickup_latitude": load.get("pick", {}).get("location", {}).get("latitude"),
                  "pickup_longitude": load.get("pick", {}).get("location", {}).get("longitude"),
                  "pickup_date": load.get("pick", {}).get("startDate"),

                  "drop_id": load.get("drop", {}).get("id", {}),
                  "drop_city": load.get("drop", {}).get("location", {}).get("city"),
                  "drop_state": load.get("drop", {}).get("location", {}).get("state"),
                  "drop_country": load.get("drop", {}).get("location", {}).get("country"),
                  "drop_latitude": load.get("drop", {}).get("location", {}).get("latitude"),
                  "drop_longitude": load.get("drop", {}).get("location", {}).get("longitude"),
                  "drop_date": load.get("drop", {}).get("startDate"),

                  "equipment_type": equipment_display_names,
                  "equipment_type_ids": equipment_type_ids,
                  "is_full_load": False if load.get("loadSize", {}).get("fullLoad")=="false" else True,
                  "load_length": load.get("loadSize", {}).get("length"),
                  "load_weight": load.get("loadSize", {}).get("weight"),
                  "load_height": load.get("loadSize", {}).get("height"),
                  "load_width": load.get("loadSize", {}).get("width"),

                  "rate": load.get("rate"),
                  "comment": load.get("comment"),

                  "contact_name": load.get("contactInfo", {}).get("dispatcherName"),
                  "contact_phone": load.get("contactInfo", {}).get("phoneNumber"),
                  "contact_email": load.get("user", {}).get("userName"),
                  "contact_fax": load.get("user", {}).get("fax"),
                  "company_name": load.get("user", {}).get("companyName"),
                  "company_email": load.get("user", {}).get("email"),
                  "MC": next((auth.get("numericValue") for auth in load.get("user", {}).get("authorities", []) if auth.get("type", {}).get("name") == "MC"), None),
                  "DOT": next((auth.get("numericValue") for auth in load.get("user", {}).get("authorities", []) if auth.get("type", {}).get("name") == "DOT"), None),
                  "estimated_distance": load.get("estimatedDistance"),
                  "hash": load.get("hash")
              })

      return extracted_data
      # print(glob.glob('loadReq/*.json'))
  else:
      print('folder does not exists')


In [9]:
def get_truckstop_data():
  extracted_data = []
  folder_path = '/content/drive/MyDrive/freight_analysis/ts_loadReq/'

  # Check if the folder exists
  if os.path.exists(folder_path):
      # Loop through all JSON files in the specified folder
      for f in glob.glob(os.path.join(folder_path, '*.json')):
          try:
              with open(f, 'r', encoding='utf-8') as file:
                  # The JSON data is an object with a "loads" key
                  data_from_file = json.load(file)
                  loads = data_from_file.get('loads', [])

          except FileNotFoundError:
              print(f"Error: The JSON file '{f}' was not found. Please ensure it's in the correct directory.")
              continue
          except json.JSONDecodeError as e:
              print(f"Error decoding JSON from '{f}': {e}")
              continue
          except Exception as e:
              print(f"An unexpected error occurred with file '{f}': {e}")
              continue

          all_processed_equipment_names = set()
          # Iterate over the list of loads
          for load in loads:
              # Find the pickup and delivery stops from the 'stops' list
              pickup_stop = next((stop for stop in load.get('stops', []) if stop.get('type') == 'Pickup'), None)
              delivery_stop = next((stop for stop in load.get('stops', []) if stop.get('type') == 'Delivery'), None)

              # Safely extract data using .get() for keys that may not exist
              pickup_address = pickup_stop.get("address", {}) if pickup_stop else {}
              delivery_address = delivery_stop.get("address", {}) if delivery_stop else {}
              equipment_info = load.get("equipment", {})
              equipment_display_names = [
                  eq_type
                  for eq_type in load.get("equipment", []).get("trailerTypes")
                  if load.get("equipment", []).get("trailerTypes") is not None # Ensure displayName exists
              ]

              # Convert equipment display names to database IDs
              equipment_type_ids = []
              for eq_name in equipment_display_names:
                  eq_id = get_or_create_equipment_id(eq_name, cursor, conn)
                  if eq_id is not None:
                      equipment_type_ids.append(eq_id)
                      all_processed_equipment_names.add(eq_name) # Add to set for testing

              # Build the dictionary for the current load with all original keys
              extracted_data.append({
                  "source": 'ts',
                  "source_id": load.get("id"),
                  "load_reference": load.get("brokerLoadId"), # Not available in new JSON
                  "post_date": load.get("createdAt"),
                  "last_updated": load.get("lastExtractedAt"),

                  "pickup_id": pickup_stop.get("stopIndex") if pickup_stop else None,
                  "pickup_city": pickup_address.get("city").title() if pickup_stop else None,
                  "pickup_state": pickup_address.get("state"),
                  "pickup_country": pickup_address.get("countryIso2"),
                  "pickup_latitude": pickup_stop.get("latitude") if pickup_stop else None,
                  "pickup_longitude": pickup_stop.get("longitude") if pickup_stop else None,
                  "pickup_date": load.get("pickup").get("appointmentStartTime") if pickup_stop else None,

                  "drop_id": delivery_stop.get("stopIndex") if delivery_stop else None,
                  "drop_city": delivery_address.get("city").title()if delivery_stop else None,
                  "drop_state": delivery_address.get("state"),
                  "drop_country": delivery_address.get("countryIso2"),
                  "drop_latitude": delivery_stop.get("latitude") if delivery_stop else None,
                  "drop_longitude": delivery_stop.get("longitude") if delivery_stop else None,
                  "drop_date": load.get("delivery").get("appointmentStartTime") if delivery_stop else None,

                  "equipment_type": equipment_info.get("trailerTypes", []),
                  "equipment_type_ids":  equipment_type_ids, # Cannot be generated without the original function
                  "is_full_load": None, # Not available in new JSON
                  "load_length": equipment_info.get("length"),
                  "load_weight": load.get("weight"),
                  "load_height": equipment_info.get("height"),
                  "load_width": equipment_info.get("width"),

                  "rate": load.get("price"),
                  'rate_per_mile':  load.get("ratePerMile"),
                  "comment": str(load.get("pickup",'').get("note",''))+', ' + str(load.get("delivery",'').get("note",'')), # Not available in new JSON
                  "contact_name": None, # Not available in new JSON
                  "contact_phone": load.get("bookingPhoneNumber"),
                  "contact_email": load.get("biddingEmail"),
                  "contact_fax": None, # Not available in new JSON
                  "company_name": load.get("broker"), # Broker name is the closest match
                  "company_email": load.get("biddingEmail"), # Closest match
                  "MC": None, # Not available in new JSON
                  "DOT": None, # Not available in new JSON
                  "estimated_distance": load.get("distance"),
                  "hash": None # Not available in new JSON
              })

      # For demonstration, let's print the first entry
      if extracted_data:
        print("Successfully extracted data from JSON files.")
        print("First extracted load:")
        # print(json.dumps(extracted_data[0], indent=2))
        print(f"\nTotal loads extracted: {len(extracted_data)}")
        return extracted_data
      else:
        print("No loads were extracted.")
  else:
      print(f"Error: The folder '{folder_path}' does not exist.")


In [10]:
relevant_deets = ['load_reference','equipment_type','load_weight','load_length','pickup','drop','pickup_zone','drop_zone','rate','estimated_distance','rate_per_mile','pickup_date','company_name','comment']
duplicate_filters = ['equipment_type','load_weight','load_length','pickup','drop','rate','pickup_date','MC','comment','post_date']

## categorize pickup and drop locations into zones using DAT map zones

In [11]:
def enrich_nextload_df(nextload_df):
  print(f'before deduplication: {len(nextload_df)}')
  nextload_df = nextload_df.drop_duplicates(subset=['hash'])
  print(f'after dropping hash: {len(nextload_df)}')
  nextload_df = nextload_df.explode('equipment_type')
  print(f'after explode: {len(nextload_df)}')

  nextload_df["pickup"] = nextload_df["pickup_city"].astype(str) + "," + nextload_df["pickup_state"].astype(str) + "," + nextload_df["pickup_country"].astype(str)
  nextload_df["drop"] = nextload_df["drop_city"].astype(str) + "," + nextload_df["drop_state"].astype(str) + "," + nextload_df["drop_country"].astype(str)
  nextload_df["lane"] = nextload_df["pickup"].astype(str) + " - " + nextload_df["drop"].astype(str)
  nextload_df['post_date'] = pd.to_datetime(nextload_df['post_date'])
  nextload_df = nextload_df.sort_values(by='post_date', ascending=True).drop_duplicates(subset=duplicate_filters, keep='last')
  nextload_df['is_repost'] = nextload_df.duplicated(subset='load_reference', keep=False)
  zone_df = pd.read_csv('/content/drive/MyDrive/freight_analysis/zone.csv')
  set_zone = zone_df.set_index('Abbreviation')['Zone']
  nextload_df['drop_zone'] = nextload_df['drop_state'].map(set_zone)
  nextload_df['pickup_zone'] = nextload_df['pickup_state'].map(set_zone)

  nextload_df['rate'] = nextload_df['rate']/100
  nextload_df['rate'] = nextload_df['rate'].astype(int)
  nextload_df['rate_per_mile'] = np.where(
  (nextload_df['rate'] == 0) | (nextload_df['estimated_distance'] == 0),
  0,
  round(nextload_df['rate'] / nextload_df['estimated_distance'], 2)
  )
  print(f'after deduplication: {len(nextload_df)}')
  nextload_df['duration_in_hours'] = nextload_df['estimated_distance']/55
  nextload_df['estimated_distance'] = nextload_df['estimated_distance'].astype(int)
  nextload_df['duration_in_driving_days'] = round(nextload_df['duration_in_hours']/10,1)
  nextload_df['duration_in_driving_days'] = nextload_df['duration_in_driving_days'].fillna(0)
  nextload_df['daily_driving_hours_left'] = nextload_df.apply(lambda row: int(max(math.ceil(row.get('duration_in_driving_days')) - round(row.get('duration_in_driving_days'),1), 0) * 10),axis=1
  )
  # Convert pickup_date column to datetime format
  nextload_df['pickup_date'] = pd.to_datetime(nextload_df['pickup_date'])
  nextload_df['drop_date'] = pd.to_datetime(nextload_df['drop_date'])

  # Floor duration and convert to timedelta
  nextload_df['estimated_delivery_date'] = nextload_df['pickup_date'] + pd.to_timedelta(nextload_df['duration_in_driving_days'].apply(math.floor), unit='D')
  nextload_df['nextload_pickup_date'] = nextload_df.apply(
      lambda row: row['estimated_delivery_date'] if row['daily_driving_hours_left'] > 2
      else row['estimated_delivery_date'] + pd.to_timedelta(1, unit='D'),
      axis=1
  )
  return nextload_df

In [12]:
def enrich_truckstop_df(truckstop_df):
  print(f'before deduplication: {len(truckstop_df)}')
  truckstop_df = truckstop_df.drop_duplicates(subset=['source_id'])
  print(f'after dropping hash: {len(truckstop_df)}')
  truckstop_df = truckstop_df.explode('equipment_type')
  print(f'after explode: {len(truckstop_df)}')


  truckstop_df["equipment_type"] = truckstop_df["equipment_type"].map({'Van':'Dry Van'})#.apply(lambda x: x.strip() if x is not None else x)
  truckstop_df["pickup_country"] = truckstop_df["pickup_country"].map({'US':'USA', 'us':'USA'})#.apply(lambda x: x.strip() if x is not None else x)
  truckstop_df["drop_country"] = truckstop_df["drop_country"].map({'US':'USA', 'us':'USA'})
  truckstop_df["pickup"] = truckstop_df["pickup_city"].astype(str) + "," + truckstop_df["pickup_state"].astype(str) + "," + truckstop_df["pickup_country"].astype(str)
  truckstop_df["drop"] = truckstop_df["drop_city"].astype(str) + "," + truckstop_df["drop_state"].astype(str) + "," + truckstop_df["drop_country"].astype(str)
  truckstop_df["lane"] = truckstop_df["pickup"].astype(str) + " - " + truckstop_df["drop"].astype(str)
  truckstop_df['post_date'] = pd.to_datetime(truckstop_df['post_date'], format='ISO8601')
  truckstop_df = truckstop_df.sort_values(by='post_date', ascending=True).drop_duplicates(subset=duplicate_filters, keep='last')
  truckstop_df['is_repost'] = truckstop_df.duplicated(subset='load_reference', keep=False)
  zone_df = pd.read_csv('/content/drive/MyDrive/freight_analysis/zone.csv')
  set_zone = zone_df.set_index('Abbreviation')['Zone']
  truckstop_df['drop_zone'] = truckstop_df['drop_state'].map(set_zone)
  truckstop_df['pickup_zone'] = truckstop_df['pickup_state'].map(set_zone)

  print(f'after deduplication: {len(truckstop_df)}')

  truckstop_df['duration_in_hours'] = truckstop_df['estimated_distance']/55
  truckstop_df['estimated_distance'] = truckstop_df['estimated_distance'].astype(int)
  truckstop_df['duration_in_driving_days'] = round(truckstop_df['duration_in_hours']/10,1)
  truckstop_df['duration_in_driving_days'] = truckstop_df['duration_in_driving_days'].fillna(0)
  truckstop_df['daily_driving_hours_left'] = truckstop_df.apply(lambda row: int(max(math.ceil(row.get('duration_in_driving_days')) - round(row.get('duration_in_driving_days'),1), 0) * 10),axis=1
  )
  # Convert pickup_date column to datetime format
  truckstop_df['pickup_date'] = pd.to_datetime(truckstop_df['pickup_date'])
  truckstop_df['drop_date'] = pd.to_datetime(truckstop_df['drop_date'])

  # Floor duration and convert to timedelta
  truckstop_df['estimated_delivery_date'] = truckstop_df['pickup_date'] + pd.to_timedelta(truckstop_df['duration_in_driving_days'].apply(math.floor), unit='D')
  truckstop_df['nextload_pickup_date'] = truckstop_df.apply(
      lambda row: row['estimated_delivery_date'] if row['daily_driving_hours_left'] > 2
      else row['estimated_delivery_date'] + pd.to_timedelta(1, unit='D'),
      axis=1
  )
  return truckstop_df

In [13]:
brokerinfo = pd.read_json('/content/drive/MyDrive/freight_analysis/ts_loadReq/broker/brokerinfo.json')


In [14]:
def rename_nextload_brokers(nextload_df, brokerinfo):
  # Remove decimal points from strings and convert to integers safely
  brokerinfo['mcNumber'] = pd.to_numeric(brokerinfo['mcNumber'], errors='coerce').fillna(0).astype(int)
  nextload_df['MC'] = pd.to_numeric(nextload_df['MC'], errors='coerce').fillna(0).astype(int)
  brokerinfo['dotNumber'] = pd.to_numeric(brokerinfo['dotNumber'], errors='coerce').fillna(0).astype(int)

  brokerinfo = brokerinfo[brokerinfo['mcNumber'] > 0][['brokerId','legalName','displayName','mcNumber','dotNumber']]
  nextload_df = nextload_df.merge(brokerinfo, how='left',left_on='MC', right_on='mcNumber')
  return nextload_df

In [15]:
def rename_truckstop_brokers(truckstop_df, brokerinfo):
  # Remove decimal points from strings and convert to integers safely
  brokerinfo['mcNumber'] = pd.to_numeric(brokerinfo['mcNumber'], errors='coerce').fillna(0).astype(int)
  brokerinfo['dotNumber'] = pd.to_numeric(brokerinfo['dotNumber'], errors='coerce').fillna(0).astype(int)

  brokerinfo = brokerinfo[['brokerId','legalName','displayName','mcNumber','dotNumber']]
  truckstop_df = truckstop_df.merge(brokerinfo, how='left', right_on='brokerId', left_on='company_name')
  return truckstop_df

In [16]:
nextload_df_original = pd.DataFrame(get_nextload_data())


In [17]:
nextload_df = nextload_df_original.copy()

In [18]:
truckstop_df_original = pd.DataFrame(get_truckstop_data())

Successfully extracted data from JSON files.
First extracted load:

Total loads extracted: 1960


In [19]:

truckstop_df = truckstop_df_original.copy()

In [20]:
nextload_df = enrich_nextload_df(nextload_df)

before deduplication: 61126
after dropping hash: 33873
after explode: 41341
after deduplication: 35861


In [21]:
truckstop_df = enrich_truckstop_df(truckstop_df)

before deduplication: 1960
after dropping hash: 1960
after explode: 2533
after deduplication: 2076


In [22]:
nextload_df = rename_nextload_brokers(nextload_df, brokerinfo)

In [23]:
truckstop_df = rename_truckstop_brokers(truckstop_df, brokerinfo)


In [24]:
nextload_df.columns

Index(['source', 'source_id', 'load_reference', 'post_date', 'last_updated',
       'pickup_id', 'pickup_city', 'pickup_state', 'pickup_country',
       'pickup_latitude', 'pickup_longitude', 'pickup_date', 'drop_id',
       'drop_city', 'drop_state', 'drop_country', 'drop_latitude',
       'drop_longitude', 'drop_date', 'equipment_type', 'equipment_type_ids',
       'is_full_load', 'load_length', 'load_weight', 'load_height',
       'load_width', 'rate', 'comment', 'contact_name', 'contact_phone',
       'contact_email', 'contact_fax', 'company_name', 'company_email', 'MC',
       'DOT', 'estimated_distance', 'hash', 'pickup', 'drop', 'lane',
       'is_repost', 'drop_zone', 'pickup_zone', 'rate_per_mile',
       'duration_in_hours', 'duration_in_driving_days',
       'daily_driving_hours_left', 'estimated_delivery_date',
       'nextload_pickup_date', 'brokerId', 'legalName', 'displayName',
       'mcNumber', 'dotNumber'],
      dtype='object')

In [25]:
truckstop_df.columns

Index(['source', 'source_id', 'load_reference', 'post_date', 'last_updated',
       'pickup_id', 'pickup_city', 'pickup_state', 'pickup_country',
       'pickup_latitude', 'pickup_longitude', 'pickup_date', 'drop_id',
       'drop_city', 'drop_state', 'drop_country', 'drop_latitude',
       'drop_longitude', 'drop_date', 'equipment_type', 'equipment_type_ids',
       'is_full_load', 'load_length', 'load_weight', 'load_height',
       'load_width', 'rate', 'rate_per_mile', 'comment', 'contact_name',
       'contact_phone', 'contact_email', 'contact_fax', 'company_name',
       'company_email', 'MC', 'DOT', 'estimated_distance', 'hash', 'pickup',
       'drop', 'lane', 'is_repost', 'drop_zone', 'pickup_zone',
       'duration_in_hours', 'duration_in_driving_days',
       'daily_driving_hours_left', 'estimated_delivery_date',
       'nextload_pickup_date', 'brokerId', 'legalName', 'displayName',
       'mcNumber', 'dotNumber'],
      dtype='object')

In [26]:
def safe_concat(df1, df2):
    # Check if both DataFrames have the same columns (names only)
    missing_in_df2 = set(df1.columns) - set(df2.columns)
    missing_in_df1 = set(df2.columns) - set(df1.columns)

    if set(df1.columns) != set(df2.columns):
        raise ValueError(
            f"❌ Column names do not match.\n"
            f"Missing in df2: {missing_in_df2}\n"
            f"Missing in df1: {missing_in_df1}"
        )
    # Check if column order is the same
    if list(df1.columns) != list(df2.columns):
        print("⚠️ Column order mismatch. Reordering df2 to match df1.")
        df2 = df2[df1.columns]

    # Concatenate safely
    return pd.concat([df1, df2], axis=0, ignore_index=True)


In [27]:
full_df = safe_concat(nextload_df, truckstop_df)

⚠️ Column order mismatch. Reordering df2 to match df1.


/tmp/ipython-input-721814157.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([df1, df2], axis=0, ignore_index=True)


In [28]:
full_df.shape

(37937, 55)

In [29]:
def ltl_filter(data, weight=20000, length=30):
  return data[(data['load_length'] < length ) & (nextload_df['load_weight'] < weight)]
ltl = ltl_filter(full_df).sort_values(by='load_length',ascending=False)


In [32]:
rate_check = nextload_df['rate']>0
rate_check.describe()

,rate
count,35861
unique,2
top,True
freq,18782


In [115]:


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great-circle distance between two points on the Earth
    (specified in decimal degrees) using the Haversine formula.
    """
    R = 3958.8  # Earth's radius in miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = R * c
    return distance

def get_load_chain(data, current, max_depth=1, filter_zones=['Z2','Z7','Z6','Z4','Z3']):
    data = data.copy()
    current = current.copy()
    depth = 1

    data['pickup_date'] = pd.to_datetime(data['pickup_date'])
    data['drop_date'] = pd.to_datetime(data['drop_date'])
    data['merge_key'] = data['pickup_state'] + '|' + data['pickup_date'].dt.strftime('%m-%d-%Y')
    zone_filtered = data[(data['pickup_zone'].isin(filter_zones) )& (data['drop_zone'].isin(filter_zones) )]
    rate_filter = zone_filtered[zone_filtered['rate_per_mile'] > 1.8].copy()
    chained = current.copy()

    while depth <= max_depth:
        previous_depth = depth - 1

        prev_suffix = f'_{previous_depth}' if depth > 1 else ''
        cur_suffix = f'_{depth}'

        if depth == 1:
            drop_date_col = 'drop_date'
            drop_state_col = 'drop_state'
            drop_lat_col = 'drop_latitude'
            drop_lon_col = 'drop_longitude'
        else:
            drop_date_col = f'drop_date{prev_suffix}'
            drop_state_col = f'drop_state{prev_suffix}'
            drop_lat_col = f'drop_latitude{prev_suffix}'
            drop_lon_col = f'drop_longitude{prev_suffix}'

        pickup_lat_col = f'pickup_latitude{cur_suffix}'
        pickup_lon_col = f'pickup_longitude{cur_suffix}'
        deadhead_distance_col = f'deadhead_distance{cur_suffix}'

        chained['merge_key'] = chained[drop_state_col] + '|' + pd.to_datetime(chained[drop_date_col]).dt.strftime('%m-%d-%Y')

        if drop_date_col not in chained.columns or drop_state_col not in chained.columns:
            print(f"[BREAK] Missing required columns at depth {depth}")
            break

        nextload_matches = chained.merge(
            rate_filter,
            how='left',
            on='merge_key',
            # left_on='merge_key',
            # right_on='merge_key',
            suffixes=('', f'_{depth}')
        )

        nextload_matches.drop(columns=['merge_key'], inplace=True)

        required_cols = [drop_lat_col, drop_lon_col, pickup_lat_col, pickup_lon_col]

        # 🎯 Fix: Use a new variable and .copy() to prevent SettingWithCopyWarning
        nextload_matches_filtered = nextload_matches.dropna(subset=required_cols).copy()

        # Use .loc to assign the new column to prevent the warning
        nextload_matches_filtered.loc[:, deadhead_distance_col] = haversine_distance(
            nextload_matches_filtered[drop_lat_col],
            nextload_matches_filtered[drop_lon_col],
            nextload_matches_filtered[pickup_lat_col],
            nextload_matches_filtered[pickup_lon_col]
        )

        chained = nextload_matches_filtered[nextload_matches_filtered[deadhead_distance_col] < 150].reset_index(drop=True).copy()

        if chained.empty:
            print(f"[BREAK] No valid matches at depth {depth}")
            break

        depth += 1

    return chained

In [103]:
unrated_data = full_df[full_df['rate'] < 1]
rated_data = full_df[full_df['rate'] > 0]
# data[['rate','rate_per_mile']]
agg_rated = rated_data.groupby(['legalName','pickup','drop']).agg({'rate':'mean','rate_per_mile':'mean','pickup_date':'size'}).reset_index()
agg_rated.shape

(8314, 6)

In [35]:
elevant_deets = ['equipment_type','load_weight','load_height','load_length','pickup','drop','rate','estimated_distance','rate_per_mile','pickup_date','company_name','comment']

# bb= rated_data[(rated_data['load_length']>0) & (rated_data['load_length']<30) & (rated_data['equipment_type']=='Dry Van') ]#[elevant_deets]
# bb.to_csv('/content/drive/MyDrive/freight_analysis/ltl_sample1.csv',index=False)
# rated_data[(rated_data['pickup_state'] == 'TX') & (rated_data['drop_state'] == 'TX')].to_csv('/content/drive/MyDrive/freight_analysis/texas_sample1.csv',index=False)


## 1. Top N Lanes by Rate Average for Equipment Type Weekly

In [37]:
import pandas as pd

# Create a simple DataFrame with a datetime index
# Notice that July 29th and 30th are missing
data = {
    'value': [10, 20, 30]
}
dates = [
    '2025-07-28', # Monday
    '2025-07-31', # Thursday
    '2025-08-01'  # Friday
]

df = pd.DataFrame(data, index=pd.to_datetime(dates))

print("--- Original DataFrame (with missing dates) ---\n")
print(df)
print("\n" + "="*50 + "\n")

print("--- Resampled to a Daily Frequency ('D') ---\n")
# The resample function creates new rows for July 29th and 30th
# and fills the 'value' with NaN since there's no data for those days.
resampled_df = df.resample('D').mean()
print(resampled_df)
print("\n" + "="*50 + "\n")

print("--- Resampled with fillna(0) ---\n")
# The .fillna(0) call from the original code would then
# convert these NaN values to 0.
resampled_df_filled = resampled_df.fillna(0)
print(resampled_df_filled)


--- Original DataFrame (with missing dates) ---

            value
2025-07-28     10
2025-07-31     20
2025-08-01     30


--- Resampled to a Daily Frequency ('D') ---

            value
2025-07-28   10.0
2025-07-29    NaN
2025-07-30    NaN
2025-07-31   20.0
2025-08-01   30.0


--- Resampled with fillna(0) ---

            value
2025-07-28   10.0
2025-07-29    0.0
2025-07-30    0.0
2025-07-31   20.0
2025-08-01   30.0


In [97]:
def analyze_top_n_lanes(dataframe: pd.DataFrame, n_days: int = 3, top_n: int = 5) -> pd.DataFrame:
    """
    Analyzes top N lanes based on average rate over the last n business days.
    """
    df = dataframe.copy()

    # Safely convert post_date to timezone-naive datetime
    df['post_date'] = pd.to_datetime(df['post_date'], utc=True).dt.tz_convert(None)

    # Filter for the last n business days
    cutoff_date = pd.to_datetime(date.today()) - pd.offsets.BDay(n_days)
    filtered_data = df[(df['post_date'].dt.weekday < 5) & (df['post_date'] >= cutoff_date)]

    # Normalize equipment types
    normalized_data = filtered_data.explode('equipment_type')

    # Add date-only column
    normalized_data['date_posted'] = normalized_data['post_date'].dt.date

    # Group and calculate metrics
    lane_metrics = normalized_data.groupby(['date_posted','pickup_state', 'drop_state', 'equipment_type']).agg(
        avg_rate=('rate_per_mile', 'mean'),
        load_count=('post_date', 'size')
    ).reset_index()

    # Sort and return top N lanes
    top_lanes = lane_metrics.sort_values(by='load_count', ascending=False)
    return top_lanes#.head(top_n)



In [98]:
    # Call the top N lanes analysis function
    top_n_lanes_result = analyze_top_n_lanes(rated_data, n_days=10, top_n=3)

In [99]:

    print("--- Top N Lanes Analysis ---")
    # print(top_n_lanes_result.to_string())
    print(top_n_lanes_result[(top_n_lanes_result['date_posted']>=(date.today()-timedelta(days=1))) & (top_n_lanes_result['load_count']>2)].sort_values(by='load_count', ascending=False).to_string())

    print("\n" + "="*50 + "\n")

--- Top N Lanes Analysis ---
     date_posted pickup_state drop_state  equipment_type  avg_rate  load_count
4615  2025-08-11           MS         TX         Flatbed  2.582241          58
4027  2025-08-11           AL         IN         Flatbed  2.391429          56
4616  2025-08-11           MS         TX       Step Deck  2.556038          53
4028  2025-08-11           AL         IN       Step Deck  2.413200          50
4016  2025-08-11           AL         GA         Flatbed  3.107442          43
4158  2025-08-11           GA         FL         Dry Van  2.529586          41
4057  2025-08-11           AL         TN         Flatbed  2.898378          37
5026  2025-08-11           TX         TX         Dry Van  4.891718          36
4018  2025-08-11           AL         GA       Step Deck  2.990323          31
4059  2025-08-11           AL         TN       Step Deck  2.861034          29
4670  2025-08-11           NC         PA         Flatbed  2.729286          28
4160  2025-08-11       

In [91]:
def analyze_weekly_trends(dataframe: pd.DataFrame, n_weeks: int = 3) -> pd.DataFrame:
    """
    Analyzes week-on-week trends by weekday for rate and load count.

    This updated version ensures every weekday (Monday-Friday) is present for every
    equipment type, filling in 0s for missing data, and returns a DataFrame with
    'equipment_type' and 'weekday' as regular columns for easier plotting.

    This version is more robust to a TypeError by delaying the Categorical conversion.
    """
    # Create a copy to prevent side effects on the original DataFrame
    df = dataframe.copy()

    # Explicitly convert to datetime before filtering and make it timezone-naive
    df['post_date'] = pd.to_datetime(df['post_date'], utc=True).dt.tz_convert(None)

    # Filter for the last n business days
    cutoff_date = pd.to_datetime(date.today()).tz_localize(None) - pd.to_timedelta(n_weeks, unit='W')
    filtered_data = df[(df['post_date'].dt.weekday < 5) & (df['post_date'] >= cutoff_date)]

    # Normalize data by exploding 'equipment_type'
    normalized_data = filtered_data.explode('equipment_type')

    # Add week number and weekday columns directly from the post_date
    normalized_data['week'] = normalized_data['post_date'].dt.isocalendar().week.astype(int)

    # --- CHANGE: Set weekday as a string. Categorical is now handled later. ---
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    normalized_data['weekday'] = normalized_data['post_date'].dt.day_name()

    # Aggregate metrics by equipment_type, weekday, and week directly
    weekly_trends = normalized_data.groupby(['equipment_type', 'weekday', 'week','lane']).agg(
        avg_rate=('rate_per_mile', 'mean'),
        load_count=('rate_per_mile', 'size')
    ).reset_index()

    # Create a DataFrame with all possible combinations of Equipment Type, Weekday, and Week
    all_equipment = normalized_data['equipment_type'].unique()
    all_weeks = normalized_data['week'].unique()
    all_lanes = normalized_data['lane'].unique()

    # --- CHANGE: Use the Categorical type here to ensure correct order in the final output ---
    all_weekdays = pd.Categorical(weekday_order, categories=weekday_order, ordered=True)

    all_combinations = pd.MultiIndex.from_product(
        [all_equipment, all_weekdays, all_weeks, all_lanes],
        names=['equipment_type', 'weekday', 'week','lane']
    ).to_frame(index=False)

    # Merge the calculated trends with the full combinations DataFrame
    merged_trends = all_combinations.merge(
        weekly_trends,
        on=['equipment_type', 'weekday', 'week', 'lane'],
        how='left'
    )

    # Fill NaN values (where there was no data) with 0
    merged_trends = merged_trends.fillna(0)
    # --- NEW: Round avg_rate to 2 decimal places before creating the pivot table ---
    merged_trends['avg_rate'] = merged_trends['avg_rate'].round(2)
    # Cast load_count to integer since it's a count
    merged_trends['load_count'] = merged_trends['load_count'].astype(int)

    # Create the pivot table
    # The `observed=False` parameter is added here to ensure all weekday categories are shown,
    # even if no data exists for them in the aggregated data.
    combined_pivot = merged_trends.pivot_table(
        index=['equipment_type', 'weekday', 'lane'],
        columns='week',
        values=['avg_rate', 'load_count'],
        observed=False
    ).fillna(0)
    load_count_cols = [col for col in combined_pivot.columns if col[0] == 'load_count']
    combined_pivot[load_count_cols] = combined_pivot[load_count_cols].astype(int)
    # --- NEW: Add a total load count column for sorting ---
    # This sums the load counts across all weeks for each row.
    combined_pivot[('total_load_count', '')] = combined_pivot[load_count_cols].sum(axis=1)
    # Flatten the multi-index to create columns for plotting
    combined_pivot = combined_pivot.reset_index()
    # combined_pivot = combined_pivot.sort_values(by='total_load_count', ascending=False)

    return combined_pivot


In [92]:
    # Call the weekly trends analysis function
dryvan = rated_data[(rated_data['equipment_type']=='Dry Van') & (rated_data['pickup_state']=='TX')]
weekly_trends_result = analyze_weekly_trends(dryvan, n_weeks=3)


In [94]:
print("--- Weekly Trends Analysis ---")
print(weekly_trends_result.sort_values(by='total_load_count', ascending=False).head(10).to_string())

--- Weekly Trends Analysis ---
     equipment_type    weekday                                   lane avg_rate                   load_count           total_load_count
week                                                                        30    31    32    33         30 31  32 33                 
1823        Dry Van    Tuesday          Orange,TX,USA - Laredo,TX,USA     1.12  0.00  1.06  0.00          1  0  10  0               11
1860        Dry Van    Tuesday     San Antonio,TX,USA - Laredo,TX,USA     0.00  0.00  1.71  0.00          0  0  10  0               10
1281        Dry Van   Thursday         Laredo,TX,USA - Memphis,TN,USA     0.00  0.00  2.14  0.00          0  0   9  0                9
1341        Dry Van   Thursday  Mount Pleasant,TX,USA - Morton,MS,USA     2.25  0.00  2.28  0.00          4  0   5  0                9
2291        Dry Van  Wednesday  Mount Pleasant,TX,USA - Morton,MS,USA     0.00  2.39  2.30  0.00          0  5   3  0                8
331         Dry Van     

In [104]:
agg_rated[agg_rated.duplicated(subset=['pickup','drop'])]

,company_name,pickup,drop,rate,rate_per_mile,pickup_date
74,A.I.G. LOGISTICS LLC,"Golden Meadow,LA,USA","Houston,TX,USA",2800.00,7.570000,1
390,Becker Logistics LLC,"Michigan City,IN,USA","Mansfield,OH,USA",750.00,2.650000,1
394,Becker Logistics LLC,"Michigan City,IN,USA","Southfield,MI,USA",7.00,0.030000,1
653,Beemac Logistics,"Dallas,TX,USA","Tulsa,OK,USA",711.00,2.570000,1
721,Beemac Logistics,"Ennis,TX,USA","Waldorf,MD,USA",2750.00,1.930000,1
...,...,...,...,...,...,...
8233,trek_transportation_brokerage,"Port Arthur,TX,USA","San Antonio,TX,USA",900.00,3.146733,1
8234,trek_transportation_brokerage,"Warren,AR,USA","Chicago,IL,USA",1725.00,2.460151,1
8235,trek_transportation_brokerage,"Zwolle,LA,USA","Tulsa,OK,USA",1075.00,2.736871,1
8250,ucw_logistics,"None,None,nan","None,None,nan",2519.97,2.432478,1


In [107]:
unique_lanes = agg_rated.groupby(['pickup', 'drop']).size().reset_index(name='count')
unique_lanes = unique_lanes.sort_values(by='count', ascending=False)
unique_lanes

,pickup,drop,count
5756,"Plain Dealing,LA,USA","Silsbee,TX,USA",8
5264,"None,None,nan","None,None,nan",7
5752,"Plain Dealing,LA,USA","Haltom City,TX,USA",6
662,"Blue Island,IL,USA","Haven,WI,USA",6
3010,"Hamburg,AR,USA","Ardmore,OK,USA",6
...,...,...,...
2632,"Fresno,CA,USA","Cheyenne,WY,USA",1
2631,"French Lick,IN,USA","Roanoke,TX,USA",1
2630,"Fremont,OH,USA","Westborough,MA,USA",1
2629,"Fremont,OH,USA","Taunton,MA,USA",1


In [108]:
agg_rated[(agg_rated['pickup']=='Laredo,TX,USA') &	(agg_rated['drop']=='Houston,TX,USA')]


,company_name,pickup,drop,rate,rate_per_mile,pickup_date
385,Becker Logistics LLC,"Laredo,TX,USA","Houston,TX,USA",8.0,0.020000,1
3138,Polaris Logistics Group,"Laredo,TX,USA","Houston,TX,USA",950.0,2.680000,1
5927,TQL,"Laredo,TX,USA","Houston,TX,USA",1000.0,2.820000,1
7846,navisphere,"Laredo,TX,USA","Houston,TX,USA",940.0,2.696579,1


In [116]:
dry_van = rated_data[pd.to_datetime(rated_data['pickup_date'],utc=True).dt.tz_convert(None).dt.date >= date.today()]
# dry_van = curr_loads[curr_loads['equipment_type'].apply( lambda row: 'Dry Van' in  row)]
pickup_texas_filter = dry_van['pickup_state'] == 'TX'
pickup_texas = dry_van[pickup_texas_filter]
pickup_texas = get_load_chain(data, pickup_texas)
pickup_texas['total_duration'] = pickup_texas.filter(like='duration_in_hours').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True)
pickup_texas['total_distance'] = pickup_texas.filter(like='distance').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True).astype(int)
pickup_texas['total_driving_hours'] = (pickup_texas['total_distance']/55).astype(int)
pickup_texas['total_revenue'] = pickup_texas.filter(like='rate').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True)
# pickup_texas['total_cost'] = pickup_texas.filter(like='rate').apply(pd.to_numeric, errors='coerce').sum(axis=1, skipna=True)
pickup_texas['total_cost_per_mile'] = pickup_texas['total_distance'] * 1.6
pickup_texas['total_profit'] = pickup_texas['total_revenue'] - pickup_texas['total_cost_per_mile']

pickup_texas.shape

KeyError: 'pickup_date'

In [117]:
dry_van.columns


Index(['source', 'source_id', 'load_reference', 'post_date', 'last_updated',
       'pickup_id', 'pickup_city', 'pickup_state', 'pickup_country',
       'pickup_latitude', 'pickup_longitude', 'pickup_date', 'drop_id',
       'drop_city', 'drop_state', 'drop_country', 'drop_latitude',
       'drop_longitude', 'drop_date', 'equipment_type', 'equipment_type_ids',
       'is_full_load', 'load_length', 'load_weight', 'load_height',
       'load_width', 'rate', 'comment', 'contact_name', 'contact_phone',
       'contact_email', 'contact_fax', 'company_name', 'company_email', 'MC',
       'DOT', 'estimated_distance', 'hash', 'pickup', 'drop', 'lane',
       'is_repost', 'drop_zone', 'pickup_zone', 'rate_per_mile',
       'duration_in_hours', 'duration_in_driving_days',
       'daily_driving_hours_left', 'estimated_delivery_date',
       'nextload_pickup_date', 'brokerId', 'legalName', 'displayName',
       'mcNumber', 'dotNumber'],
      dtype='object')

In [ ]:
trip_result = pickup_texas[['total_distance','total_revenue','total_cost_per_mile','total_profit']]
trip_result

,total_distance,total_revenue,total_cost_per_mile,total_profit
0,1933,4035.993221,3092.8,943.193221
1,2087,3859.549824,3339.2,520.349824
2,2087,3859.549824,3339.2,520.349824
3,2245,4209.526157,3592.0,617.526157
4,2245,4209.526157,3592.0,617.526157
...,...,...,...,...
1019,2245,4209.526157,3592.0,617.526157
1020,2087,3859.549824,3339.2,520.349824
1021,2087,3859.549824,3339.2,520.349824
1022,2245,4209.526157,3592.0,617.526157


In [ ]:
profitable = trip_result[trip_result['total_profit'] > 0 ].sort_values(by='total_profit', ascending=False)
print(profitable.describe())
print(profitable.head(30))


       total_distance  total_revenue  total_cost_per_mile  total_profit
count     1024.000000    1024.000000          1024.000000   1024.000000
mean      2113.081055    4028.909984          3380.929688    647.980296
std        357.049542     615.436159           571.279268    255.463865
min        590.000000    1469.462985           944.000000    364.898612
25%       2087.000000    3859.549824          3339.200000    495.007091
50%       2230.000000    4085.597991          3568.000000    578.088100
75%       2245.000000    4209.526157          3592.000000    662.917152
max       2644.000000    5261.930021          4230.400000   1441.391909
     total_distance  total_revenue  total_cost_per_mile  total_profit
60             1897    4476.591909               3035.2   1441.391909
988            1897    4476.591909               3035.2   1441.391909
108            1897    4476.591909               3035.2   1441.391909
940            1897    4476.591909               3035.2   1441.391909
39

In [ ]:
# non_profitable = trip_result['total_profit'] < 0
non_profitable = trip_result[trip_result['total_profit'] < 0 ]
print(non_profitable.describe())
print(non_profitable)

       total_distance  total_revenue  total_cost_per_mile  total_profit
count             0.0            0.0                  0.0           0.0
mean              NaN            NaN                  NaN           NaN
std               NaN            NaN                  NaN           NaN
min               NaN            NaN                  NaN           NaN
25%               NaN            NaN                  NaN           NaN
50%               NaN            NaN                  NaN           NaN
75%               NaN            NaN                  NaN           NaN
max               NaN            NaN                  NaN           NaN
Empty DataFrame
Columns: [total_distance, total_revenue, total_cost_per_mile, total_profit]
Index: []


In [ ]:
ftr = pickup_texas[pickup_texas['total_profit'] > 1000 ].filter(regex='state|city|zone|distance|rate|total|date|hash', axis=1)
ftr

,post_date,last_updated,pickup_city,pickup_state,pickup_date,drop_city,drop_state,drop_date,rate,estimated_distance,hash,drop_zone,pickup_zone,rate_per_mile,estimated_delivery_date,nextload_pickup_date,post_date_1,last_updated_1,pickup_city_1,pickup_state_1,pickup_date_1,drop_city_1,drop_state_1,drop_date_1,rate_1,estimated_distance_1,hash_1,drop_zone_1,pickup_zone_1,rate_per_mile_1,estimated_delivery_date_1,nextload_pickup_date_1,deadhead_distance_1,post_date_2,last_updated_2,pickup_city_2,pickup_state_2,pickup_date_2,drop_city_2,drop_state_2,drop_date_2,rate_2,estimated_distance_2,hash_2,drop_zone_2,pickup_zone_2,rate_per_mile_2,estimated_delivery_date_2,nextload_pickup_date_2,deadhead_distance_2,post_date_3,last_updated_3,pickup_city_3,pickup_state_3,pickup_date_3,drop_city_3,drop_state_3,drop_date_3,rate_3,estimated_distance_3,hash_3,drop_zone_3,pickup_zone_3,rate_per_mile_3,estimated_delivery_date_3,nextload_pickup_date_3,deadhead_distance_3,total_duration,total_distance,total_driving_hours,total_revenue,total_cost_per_mile,total_profit
9,2025-08-05T06:20:09-05:00,2025-08-07T16:35:06-05:00,Orange,TX,2025-08-08,Carville,LA,2025-08-08,700,188,d9715913db36eed15d1e7b314e83236d,Z7,Z7,3.723404,2025-08-08,2025-08-08,2025-08-05T10:40:11-05:00,2025-08-07T16:35:06-05:00,Lacombe,LA,2025-08-08,Hickory,NC,2025-08-11,1350.0,746.0,d4b88ed9cd490f91d2c7ca21d07c9fc8,Z2,Z7,1.809651,2025-08-09,2025-08-09,69.106857,2025-08-07T15:45:54-05:00,2025-08-07T15:45:55-05:00,Elkin,NC,2025-08-11,Millersburg,OH,2025-08-12,975.0,364.0,9dbf4927a8d164b09f07064dc147f4fc,Z4,Z2,2.678571,2025-08-11,2025-08-11,44.833402,2025-08-05T09:00:16-05:00,2025-08-07T16:35:06-05:00,Akron,OH,2025-08-12,Statesville,NC,2025-08-13,1050.0,440.0,0c95ddaf61c2939e79ee09db50ebe84e,Z2,Z4,2.386364,2025-08-12,2025-08-13,41.960198,31.600000,1893,34,4085.597991,3028.8,1056.797991
12,2025-08-05T06:20:09-05:00,2025-08-07T16:35:06-05:00,Orange,TX,2025-08-08,Carville,LA,2025-08-08,700,188,d9715913db36eed15d1e7b314e83236d,Z7,Z7,3.723404,2025-08-08,2025-08-08,2025-08-05T10:40:11-05:00,2025-08-07T16:35:06-05:00,Lacombe,LA,2025-08-08,Hickory,NC,2025-08-11,1350.0,746.0,d4b88ed9cd490f91d2c7ca21d07c9fc8,Z2,Z7,1.809651,2025-08-09,2025-08-09,69.106857,2025-08-07T15:45:54-05:00,2025-08-07T15:45:55-05:00,Elkin,NC,2025-08-11,Millersburg,OH,2025-08-12,975.0,364.0,9dbf4927a8d164b09f07064dc147f4fc,Z4,Z2,2.678571,2025-08-11,2025-08-11,44.833402,2025-08-07T15:19:10-05:00,2025-08-07T15:19:10-05:00,Norwalk,OH,2025-08-12,Edgewood,MD,NaT,1440.0,426.0,552c8736151c7fa4336267808c2481d8,Z2,Z4,3.380282,2025-08-12,2025-08-13,59.901228,31.345455,1897,34,4476.591909,3035.2,1441.391909
13,2025-08-05T06:20:09-05:00,2025-08-07T16:35:06-05:00,Orange,TX,2025-08-08,Carville,LA,2025-08-08,700,188,d9715913db36eed15d1e7b314e83236d,Z7,Z7,3.723404,2025-08-08,2025-08-08,2025-08-05T10:40:11-05:00,2025-08-07T16:35:06-05:00,Lacombe,LA,2025-08-08,Hickory,NC,2025-08-11,1350.0,746.0,d4b88ed9cd490f91d2c7ca21d07c9fc8,Z2,Z7,1.809651,2025-08-09,2025-08-09,69.106857,2025-08-07T15:45:54-05:00,2025-08-07T15:45:55-05:00,Elkin,NC,2025-08-11,Millersburg,OH,2025-08-12,975.0,364.0,9dbf4927a8d164b09f07064dc147f4fc,Z4,Z2,2.678571,2025-08-11,2025-08-11,44.833402,2025-08-07T15:25:12-05:00,2025-08-07T15:25:12-05:00,Alliance,OH,2025-08-12,Auburn,IN,NaT,880.0,250.0,b7d1c32eb46acd4be999371730f7c85b,Z4,Z4,3.520000,2025-08-12,2025-08-12,49.282683,28.145455,1711,31,3916.731627,2737.6,1179.131627
14,2025-08-05T06:20:09-05:00,2025-08-07T16:35:06-05:00,Orange,TX,2025-08-08,Carville,LA,2025-08-08,700,188,d9715913db36eed15d1e7b314e83236d,Z7,Z7,3.723404,2025-08-08,2025-08-08,2025-08-05T10:40:11-05:00,2025-08-07T16:35:06-05:00,Lacombe,LA,2025-08-08,Hickory,NC,2025-08-11,1350.0,746.0,d4b88ed9cd490f91d2c7ca21d07c9fc8,Z2,Z7,1.809651,2025-08-09,2025-08-09,69.106857,2025-08-07T15:45:54-05:00,2025-08-07T15:45:55-05:00,Elkin,NC,2025-08-11,Millersburg,OH,2025-08-12,975.0,364.0,9dbf4927a8d164b09f07064dc147f4fc,Z4,Z2,2.678571,2025-08-11,2025-08-11,44.833402,2025-08-05T09:00:

In [ ]:
grouped = ftr.groupby('drop_zone_3').agg(
    total_distance=('total_distance', 'mean'),
    total_revenue=('total_revenue', 'mean'),
    total_cost_per_mile=('total_cost_per_mile', 'mean'),
    total_profit=('total_profit', 'mean')
).rename(
    columns={
        'total_distance': 'average_distance',
        'total_revenue': 'average_revenue',
        'total_cost_per_mile': 'average_cost_per_mile',
        'total_profit': 'average_profit'
    }
).reset_index()

grouped

,drop_zone_3,average_distance,average_revenue,average_cost_per_mile,average_profit
0,Z2,2108.540541,4504.424435,3373.664865,1130.759570
1,Z3,1817.000000,3938.849407,2907.200000,1031.649407
2,Z4,1735.052632,3902.925595,2776.084211,1126.841384
3,Z6,2423.666667,5162.080156,3877.866667,1284.213489


In [ ]:
fig = px.scatter(ftr, x='total_driving_hours', y='total_profit',
                 trendline='ols',
                 title='driving hours by total_profit')
fig.show()

In [ ]:
ftr[(ftr['total_profit']<3000) & (ftr['total_profit']>1800 & (ftr['total_driving_hours'] < 45))]

,pickup_city,pickup_state,drop_city,drop_state,rate,estimated_distance,drop_zone,pickup_zone,rate_per_mile,pickup_city_1,pickup_state_1,drop_city_1,drop_state_1,rate_1,estimated_distance_1,drop_zone_1,pickup_zone_1,rate_per_mile_1,deadhead_distance_1,pickup_city_2,pickup_state_2,drop_city_2,drop_state_2,rate_2,estimated_distance_2,drop_zone_2,pickup_zone_2,rate_per_mile_2,deadhead_distance_2,pickup_city_3,pickup_state_3,drop_city_3,drop_state_3,rate_3,estimated_distance_3,drop_zone_3,pickup_zone_3,rate_per_mile_3,deadhead_distance_3,total_duration,total_distance,total_driving_hours,total_revenue,total_cost_per_mile,total_profit
674,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Mitchell,IN,Oneida,KY,750.0,220.0,Z4,Z4,3.409091,12.209766,33.490909,1975,35,5462.955825,3160.0,2302.955825
675,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Alexandria,IN,Hickory,NC,1100.0,469.0,Z2,Z4,2.345416,125.277849,38.018182,2337,42,5811.892150,3739.2,2072.692150
676,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Alexandria,IN,Hickory,NC,1100.0,469.0,Z2,Z4,2.345416,125.277849,38.018182,2337,42,5811.892150,3739.2,2072.692150
677,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Ferdinand,IN,Mauldin,NC,1100.0,502.0,Z2,Z4,2.191235,31.352720,38.618182,2277,41,5811.737969,3643.2,2168.537969
678,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Lafayette,IN,Garden City,TX,2400.0,1205.0,Z7,Z4,1.991701,130.370466,51.400000,3079,55,7111.538435,4926.4,2185.138435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19560,Laredo,TX,Battle Creek,MI,2800,1562,Z4,Z7,1.792574,Kalamazoo,MI,Suwanee,GA,1750.0,690.0,Z3,Z4,2.536232,20.922705,Augusta,GA,Pineville,NC,500.0,154.0,Z2,Z3,3.246753,124.926196,Browns Summit,NC,Northbrook,IL,1400.0,714.0,Z6,Z2,1.960784,102.342741,56.727273,3368,61,6459.536343,5388.8,1070.736343
19566,Laredo,TX,Battle Creek,MI,2800,1562,Z4,Z7,1.792574,Kalamazoo,MI,Suwanee,GA,1750.0,690.0,Z3,Z4,2.536232,20.922705,Augusta,GA,Pineville,NC,500.0,154.0,Z2,Z3,3.246753,124.926196,Browns Summit,NC,Northbrook,IL,1400.0,714.0,Z6,Z2,1.960784,102.342741,56.727273,3368,61,6459.536343,5388.8,1070.736343
19567,Laredo,TX,Battle Creek,MI,2800,1562,Z4,Z7,1.792574,Kalamazoo,MI,Suwanee,GA,1750.0,690.0,Z3,Z4,2.536232,20.922705,Augusta,GA,Pineville,NC,500.0,154.0,Z2,Z3,3.246753,124.926196,Browns Summit,NC,Northbrook,IL,1400.0,714.0,Z6,Z2,1.960784,102.342741,56.727273,3368,61,6459.536343,5388.8,1070.736343
19573,Laredo,TX,Battle Creek,MI,2800,1562,Z4,Z7,1.792574,Kalamazoo,MI,Suwanee,GA,1750.0,690.0,Z3,Z4,2.536232,20.922705,Augusta,GA,Pineville,NC,500.0,154.0,Z2,Z3,3.246753,124.926196,Browns Summit,NC,Northbrook,IL,1400.0,714.0,Z6,Z2,1.960784,102.342741,56.727273,3368,61,6459.536343,5388.8,1070.736343


In [ ]:
print(ftr.groupby(['nextload_pickup_date','pickup_state_1'])['hash_1'].nunique())#.agg({'drop_state':'size'}).reset_index())
print(ftr.groupby(['nextload_pickup_date_1','pickup_state_2'])['hash_2'].nunique())
print(ftr.groupby(['nextload_pickup_date_2','pickup_state_3'])['hash_3'].nunique())

nextload_pickup_date  pickup_state_1
2025-08-04            LA                 2
                      OH                15
                      TX                 3
2025-08-05            LA                 2
                      TX                 4
2025-08-06            AR                 5
                      LA                 4
                      TX                 5
2025-08-07            NC                 1
                      TX                 1
2025-08-08            MI                 4
                      TN                 1
                      TX                 1
2025-08-09            MI                 4
                      TN                 1
Name: hash_1, dtype: int64
nextload_pickup_date_1  pickup_state_2
2025-08-05              LA                 1
                        TX                32
2025-08-06              AR                25
                        IN                 4
                        OH                 3
                        SC 

In [ ]:
# Correctly filter the DataFrame where the 'drop_zone_1' column
# contains any of the values in the list
filtered_df = ftr[pickup_texas['drop_zone_3'].isin(['Z2','Z7','Z6','Z4','Z3'])]
filtered_df

/tmp/ipython-input-2439403874.py:3: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



,pickup_city,pickup_state,drop_city,drop_state,rate,estimated_distance,drop_zone,pickup_zone,rate_per_mile,pickup_city_1,pickup_state_1,drop_city_1,drop_state_1,rate_1,estimated_distance_1,drop_zone_1,pickup_zone_1,rate_per_mile_1,deadhead_distance_1,pickup_city_2,pickup_state_2,drop_city_2,drop_state_2,rate_2,estimated_distance_2,drop_zone_2,pickup_zone_2,rate_per_mile_2,deadhead_distance_2,pickup_city_3,pickup_state_3,drop_city_3,drop_state_3,rate_3,estimated_distance_3,drop_zone_3,pickup_zone_3,rate_per_mile_3,deadhead_distance_3,total_duration,total_distance,total_driving_hours,total_revenue,total_cost_per_mile,total_profit
842,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Mitchell,IN,Oneida,KY,750.0,220.0,Z4,Z4,3.409091,12.209766,33.490909,1975,35,5462.955825,3160.0,2302.955825
843,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Alexandria,IN,Hickory,NC,1100.0,469.0,Z2,Z4,2.345416,125.277849,38.018182,2337,42,5811.892150,3739.2,2072.692150
844,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Alexandria,IN,Hickory,NC,1100.0,469.0,Z2,Z4,2.345416,125.277849,38.018182,2337,42,5811.892150,3739.2,2072.692150
845,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Ferdinand,IN,Mauldin,NC,1100.0,502.0,Z2,Z4,2.191235,31.352720,38.618182,2277,41,5811.737969,3643.2,2168.537969
846,La Porte,TX,Geismar,LA,0,292,Z7,Z7,0.000000,Golden Meadow,LA,Houston,TX,2800.0,370.0,Z7,Z7,7.567568,73.302804,Hempstead,TX,Paoli,IN,1900.0,960.0,Z4,Z7,1.979167,48.394550,Lafayette,IN,Garden City,TX,2400.0,1205.0,Z7,Z4,1.991701,130.370466,51.400000,3079,55,7111.538435,4926.4,2185.138435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27780,Port Arthur,TX,Temple,TX,925,263,Z7,Z7,3.517110,Fort Worth,TX,Alorton,IL,4700.0,684.0,Z6,Z7,6.871345,114.508632,St. Elmo,IL,Fort Worth,TX,1500.0,770.0,Z7,Z6,1.948052,74.859500,Rockwall,TX,Haines City,FL,2700.0,1130.0,Z3,Z7,2.389381,52.002164,51.763636,3088,56,9839.725888,4940.8,4898.925888
27781,Port Arthur,TX,Temple,TX,925,263,Z7,Z7,3.517110,Fort Worth,TX,Alorton,IL,4700.0,684.0,Z6,Z7,6.871345,114.508632,St. Elmo,IL,Fort Worth,TX,1500.0,770.0,Z7,Z6,1.948052,74.859500,Garland,TX,Seffner,FL,2700.0,1103.0,Z3,Z7,2.447869,41.609268,51.272727,3050,55,9839.784377,4880.0,4959.784377
27806,Port Arthur,TX,Comanche,TX,1075,371,Z7,Z7,2.897574,Fort Worth,TX,Alorton,IL,4700.0,684.0,Z6,Z7,6.871345,95.070328,St. Elmo,IL,Fort Worth,TX,1500.0,770.0,Z7,Z6,1.948052,74.859500,Rockwall,TX,Haines City,FL,2700.0,1130.0,Z3,Z7,2.389381,52.002164,53.727273,3176,57,9989.106352,5081.6,4907.506352
27807,Port Arthur,TX,Comanche,TX,1075,371,Z7,Z7,2.897574,Fort Worth,TX,Alorton,IL,4700.0,684.0,Z6,Z7,6.871345,95.070328,St. Elmo,IL,Fort Worth,TX,1500.0,770.0,Z7,Z6,1.948052,74.859500,Garland,TX,Seffner,FL,2700.0,1103.0,Z3,Z7,2.447869,41.609268,53.236364,3139,57,9989.164841,5022.4,4966.764841


In [ ]:
pickup_texas.filter(like='deadhead')

,deadhead_distance_1
0,0.000000
1,115.490688
2,23.600989
3,23.600989
4,14.501675
...,...
4707,32.629187
4708,93.170356
4709,32.629187
4710,54.314857


In [ ]:
backhaul = pickup_texas.groupby(['hash', 'pickup_state_1']).agg({'hash_1':'size'}).reset_index().rename(columns={'hash_1':'count'})
# result = backhaul.merge(pickup_texas[['hash', 'drop_state_1']], on='hash', how='left')
backhaul



,hash,pickup_state_1,count
0,0013b14839aaae5a82f26c69f5d81783,TX,127
1,0036719b3884447fbdb6e4b94d2c7b87,TX,23
2,004718e1d34bc66cf54f005b64677f48,LA,51
3,008218287d8cd94ac52673fd596e6674,TX,62
4,0095b0bc202532aa193ec105fa54c29a,FL,28
...,...,...,...
1477,fe7739e628a84b188cc2a25a150328ba,TX,302
1478,fe9495f2c5355577b39fc8a4bd3a766c,TX,108
1479,fea9c66c67f4bfb2bb8f7bffd917a8f6,TX,302
1480,fece79e664d031665313b950092d79ca,TX,302


In [ ]:
merge_keys_set = set(data['merge_key'].values)

for _, rw in pickup_texas.iterrows():
    if rw['merge_key'] in merge_keys_set:
      print('truuu')

truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truuu
truu

In [ ]:
unique_states = numpy.unique(numpy.append(data['pickup_state'].unique(),data['drop_state'].unique().tolist()))
us = pd.DataFrame(unique_states, columns=['state'])
drop_state_counts = data['drop_state'].value_counts()
pickup_state_counts = data['pickup_state'].value_counts()
us = pd.concat([pickup_state_counts, drop_state_counts], axis=1).fillna(0).astype(int)
us.columns = ['pickup', 'drop']
us

In [ ]:
cursor.execute("SELECT equipment FROM equipments")
equipment_list = cursor.fetchall()
equipment_list

In [ ]:
import pandas as pd
import numpy
def pick_n_drop(data):
# Filter loads with rate_per_mile > 2
  high_value_loads = data[data['rate_per_mile'] > 2]

  # Get distinct pickup and drop states
  distinct_states = numpy.unique(
      numpy.append(
          high_value_loads['pickup_state'].unique(),
          high_value_loads['drop_state'].unique()
      )
  )
  state_df = pd.DataFrame(distinct_states, columns=['state'])

  # Count pickups and drops
  pickup_counts = high_value_loads['pickup_state'].value_counts()
  drop_counts = high_value_loads['drop_state'].value_counts()

  # Combine the counts
  state_summary = pd.concat([pickup_counts, drop_counts], axis=1).fillna(0).astype(int)

  # Rename columns for clarity
  state_summary.columns = ['pickup_total', 'drop_total']

  return state_summary

In [ ]:
# flag reposted postings
data['is_repost'] = data.duplicated(subset='load_reference', keep=False)

# count how many times each load was posted
post_counts = (
    data.groupby('load_reference')
      .size()
      .rename('num_posts')
      .reset_index()
)
post_counts[post_counts['num_posts']>1]

# # merge back to the main DF
# df = df.merge(post_counts, on='load_reference')

,load_reference,num_posts
4,-1657337293,2
8,-1885402074,2
132,0299476,2
133,0299540,2
149,0299852,2
...,...,...
3528,aC0Pg000003bAzZKAU,2
3535,aC0Pg000003bTqvKAE,3
3551,aC0Pg000003c8E9KAI,2
3574,aC0Pg000003dXDhKAM,2


In [ ]:


data['post_date']   = pd.to_datetime(data['post_date'],   utc=True)
data['pickup_date'] = pd.to_datetime(data['pickup_date'],   utc=True)
data['last_updated']= pd.to_datetime(data['last_updated'],   utc=True)

# compute lead time in days
data['lead_days'] = (data['pickup_date'] - data['post_date']).dt.days.clip(lower=0)

In [ ]:
# summary of lead days
print(data['lead_days'].describe())

# summary of rate per mile
print(data['rate_per_mile'].describe())

# count shipments by lead_days
lead_counts = data['lead_days'].value_counts().sort_index()
print(lead_counts.head(10))

count    3769.000000
mean        1.499071
std         3.412317
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        56.000000
Name: lead_days, dtype: float64
count    3769.000000
mean        1.288655
std         1.982242
min         0.000000
25%         0.000000
50%         0.000000
75%         2.384292
max        57.142857
Name: rate_per_mile, dtype: float64
lead_days
0    2313
1     525
2     277
3     151
4      82
5     127
6      83
7      36
8      23
9      22
Name: count, dtype: int64


In [ ]:
# dry_van = data['Dry Van' in data['equipment_type']]
dry_van = data[data['equipment_type'].apply( lambda row: 'Dry Van' in  row)]
rated_dry_van = data[data['rate_per_mile'] >0]
dry_van[relevant_deets].sample(10)

,load_reference,equipment_type,load_weight,load_length,pickup,drop,pickup_zone,drop_zone,rate,estimated_distance,rate_per_mile,pickup_date,company_name,comment
2418,1819008,[Dry Van],9152,53,"Plant City,FL,USA","El Dorado,AR,USA",Z3,Z7,1100,866,1.270208,2025-08-07 00:00:00+00:00,Beemac Logistics,V53 w/ swing doors 10k lbs need load bars or straps
4406,aC0Pg000003dzwLKAQ,[Dry Van],16461,0,"Waller,TX,USA","Amarillo,TX,USA",Z7,Z7,0,594,0.000000,2025-07-29 00:00:00+00:00,Werner,None
4149,0293857,[Dry Van],43280,0,"Dalhart,TX,USA","Richland Cente,WI,None",Z7,Z5,0,979,0.000000,2025-07-30 00:00:00+00:00,Tumalo Creek Transportation,None
3761,aC0Pg000003dwqPKAQ,[Dry Van],42000,0,"Laredo,TX,USA","Laredo,TX,USA",Z7,Z7,0,0,0.000000,2025-07-29 00:00:00+00:00,Werner,None
1861,1837350,[Dry Van],39744,53,"Dalton,GA,USA","Orlando,FL,USA",Z3,Z3,1500,568,2.640845,2025-08-06 00:00:00+00:00,Beemac Logistics,"V53 w/ swing doors need 5 straps, 42500 lbs"
3134,9801229,[Dry Van],45000,0,"Lawrenceburg,KY,USA","Osceola,AR,USA",Z4,Z7,750,388,1.932990,2025-08-07 00:00:00+00:00,Sureway Transportation Co / Anderson Trucking Serv,3 DAY PAY NO FEE
4226,719918,[Dry Van],40000,53,"Houston,TX,USA","Baton Rouge,LA,USA",Z7,Z7,0,305,0.000000,2025-07-28 00:00:00+00:00,BBI Logistics,Ext:514 |
3896,16003094,[Dry Van],44000,0,"Brownsville,TX,USA","Richmond,VA,USA",Z7,Z2,0,1706,0.000000,2025-07-28 00:00:00+00:00,DSV Road,"V53\nPLASTIC PRODUCTS\nBr40-Laredo, TX"
157,57412037,[Dry Van],6485,0,"Conroe,TX,USA","Fort Worth,TX,USA",Z7,Z7,0,236,0.000000,2025-08-07 00:00:00+00:00,CRST Logistics Inc,None
3696,7157346,[Dry Van],26656,0,"Laredo,TX,USA","Sumter,SC,USA",Z7,Z2,4800,1462,3.283174,2025-07-28 00:00:00+00:00,TQL,None


In [ ]:


fig = px.histogram(dry_van, x='lead_days',
                   nbins=15,
                   title='Distribution of Lead Time (Days)')
fig.show()

In [ ]:
fig = px.scatter(rated_dry_van, x='lead_days', y='rate_per_mile',
                 trendline='ols',
                 title='Rate per Mile by Lead Days')
fig.show()

In [ ]:
fig = px.scatter(rated_dry_van, x='estimated_distance', y='rate_per_mile',
                 trendline='ols',
                 title='Rate per Mile by estimated distance')
fig.show()

In [ ]:
fig = px.scatter(rated_dry_van[rated_dry_van['load_weight'] < 50000], x='load_weight', y='rate_per_mile',
                 trendline='ols',
                 title='Rate per Mile by weight')
fig.show()

In [ ]:
import plotly.express as px

fig = px.histogram(rated_dry_van[rated_dry_van['load_weight'] < 48000], x='load_weight',
                   nbins=15,
                   title='Distribution of Lead Time (Days)')
fig.show()

In [ ]:
# flag reposted postings
data['is_repost'] = data.duplicated(subset='load_reference', keep=False)

# count how many times each load was posted
post_counts = (
    data.groupby('load_reference')
      .size()
      .rename('num_posts')
      .reset_index()
)

# merge back to the main data
data = data.merge(post_counts, on='load_reference')

In [ ]:
metrics = (
    data.groupby('is_repost')
      .agg(
          count_loads=('load_reference','nunique'),
          avg_rate=('rate_per_mile','mean'),
          avg_lead_days=('lead_days','mean')
      )
      .reset_index()
)
print(metrics)

   is_repost  count_loads  avg_rate  avg_lead_days
0      False         3576  1.279692       1.328300
1       True           95  1.454722       4.663212


In [ ]:
# sort posts
reposts = data[data['is_repost']].sort_values(
    ['load_reference','post_date']
)

# compute time to next post
reposts['next_post_date'] = reposts.groupby('load_reference')['post_date'].shift(-1)
reposts['days_to_repost'] = (
    reposts['next_post_date'] - reposts['post_date']
).dt.days

# distribution of repost intervals
print(reposts['days_to_repost'].describe())

count    98.000000
mean      2.765306
std       5.960006
min       0.000000
25%       0.000000
50%       0.000000
75%       1.750000
max      27.000000
Name: days_to_repost, dtype: float64


In [ ]:
# calculate sequence number for each post
reposts['post_seq'] = reposts.groupby('load_reference').cumcount() + 1

fig = px.line(
  reposts, x='post_seq', y='rate_per_mile',
  color='load_reference',
  title='Rate Trajectory Across Reposts'
)
fig.show()

In [ ]:
data.to_csv('/content/drive/MyDrive/freight_analysis/samp.csv')